# Qualtrics → DataFrame (All 6 Surveys)
Fetches raw response data for the 6 Qualtrics surveys used in the Qlik `NPS_Score` load script (`Qualtrics_Data_Extract` app):

| Survey | Survey ID |
|---|---|
| Customer Satisfaction Survey (CSAT) | `SV_eXLP1SmIM6zjdXv` |
| POC Survey | `SV_9GjRYiNTNKsuKkR` |
| Health Services | `SV_1Yx80MMG0nDHTAF` |
| Mental Health Program | `SV_2hm4kwhNudIHB2t` |
| Health and Wellbeing Program | `SV_6m5bFDluaJjxO8R` |
| HCS NPS Questionnaire (Wellbeing V2) | `SV_efX6mORlYWtebXg` |

Each survey's raw export is fetched via the Qualtrics API (initiate export → poll → download zip → extract CSV), skipping the two Qlik/Qualtrics metadata rows (question text, ImportId) after the header.

In [ ]:
import requests
import pandas as pd
import zipfile
import io
import time

API_TOKEN = "lyf0SLF0sEvAgCx6t94UskYsmiTBfWt7ADZ3dnwn"
BASE_URL  = "https://syd1.qualtrics.com/API/v3"

SURVEYS = {
    "CSAT":         "SV_eXLP1SmIM6zjdXv",  # Customer Satisfaction Survey
    "POC":          "SV_9GjRYiNTNKsuKkR",  # POC Survey
    "HealthServices": "SV_1Yx80MMG0nDHTAF",  # Health Services
    "MentalHealth": "SV_2hm4kwhNudIHB2t",  # Mental Health Program
    "Wellbeing":    "SV_6m5bFDluaJjxO8R",  # Health and Wellbeing Program
    "WellbeingV2":  "SV_efX6mORlYWtebXg",  # HCS NPS Questionnaire (Wellbeing V2)
}

headers = {"X-API-TOKEN": API_TOKEN}

In [ ]:
def fetch_survey_responses(survey_id: str, label: str) -> pd.DataFrame:
    # Step 1: Initiate export
    resp = requests.post(
        f"{BASE_URL}/surveys/{survey_id}/export-responses",
        headers=headers,
        json={"format": "csv", "useLabels": True}
    )
    resp.raise_for_status()
    progress_id = resp.json()["result"]["progressId"]
    print(f"[{label}] Export started — progressId: {progress_id}")

    # Step 2: Poll until complete
    file_id = None
    while True:
        poll = requests.get(
            f"{BASE_URL}/surveys/{survey_id}/export-responses/{progress_id}",
            headers=headers
        )
        poll.raise_for_status()
        result = poll.json()["result"]
        status = result["status"]
        if status == "complete":
            file_id = result["fileId"]
            break
        elif status == "failed":
            raise RuntimeError(f"[{label}] Export failed")
        time.sleep(2)

    print(f"[{label}] Done — fileId: {file_id}")

    # Step 3: Download zip → extract CSV → raw DataFrame
    download = requests.get(
        f"{BASE_URL}/surveys/{survey_id}/export-responses/{file_id}/file",
        headers=headers
    )
    download.raise_for_status()

    with zipfile.ZipFile(io.BytesIO(download.content)) as z:
        csv_name = z.namelist()[0]
        # Qualtrics CSVs: row 0 = col names, row 1 = question text, row 2 = ImportId
        df = pd.read_csv(z.open(csv_name), skiprows=[1, 2], low_memory=False)

    print(f"[{label}] Raw rows: {len(df):,}  |  Columns: {len(df.columns)}")
    return df

In [ ]:
# Fetch all 5 surveys into a dict of raw DataFrames
raw_dfs = {}
for label, survey_id in SURVEYS.items():
    raw_dfs[label] = fetch_survey_responses(survey_id, label)

In [ ]:
# Summary of all fetched surveys
for label, df in raw_dfs.items():
    print(f"{label:15s} rows={len(df):>6,}  cols={len(df.columns)}")

In [ ]:
# Preview CSAT (Customer Satisfaction Survey) raw data + full column list
print(list(raw_dfs["CSAT"].columns))
raw_dfs["CSAT"].head()

In [ ]:
# Preview the other 5 surveys' raw columns
for label in ["POC", "HealthServices", "MentalHealth", "Wellbeing", "WellbeingV2"]:
    print(f"\n=== {label} ===")
    print(list(raw_dfs[label].columns))

# Survey question definitions (ground truth for field meanings)
Qlik script comments reference `QID` numbers that repeatedly don't match the actual API export field names (`Q2`, `Q7`, etc.), and question numbering can drift whenever a survey is edited. Instead of guessing from data content, query Qualtrics' `survey-definitions` API directly to see each survey's current question text mapped to its export field ID.

In [ ]:
def fetch_question_definitions(survey_id: str) -> pd.DataFrame:
    resp = requests.get(
        f"{BASE_URL}/survey-definitions/{survey_id}/questions",
        headers=headers
    )
    resp.raise_for_status()
    elements = resp.json()["result"]["elements"]

    rows = []
    for q in elements:
        rows.append({
            "QuestionID": q.get("QuestionID"),          # e.g. QID3 — Qlik's numbering
            "DataExportTag": q.get("DataExportTag"),     # e.g. Q3 — the actual CSV/API export field name
            "QuestionType": q.get("QuestionType"),
            "Selector": q.get("Selector"),
            "QuestionText": q.get("QuestionText"),
        })
    return pd.DataFrame(rows)


question_defs = {}
for label, survey_id in SURVEYS.items():
    question_defs[label] = fetch_question_definitions(survey_id)
    print(f"[{label}] {len(question_defs[label])} questions")

# NPS_Score (replicates Qlik `NPS_Score` table)
Standardises each survey's raw columns to the common field names used in the Qlik load script, then concatenates all 6 surveys into one table, keeping only rows with a non-empty Member Number.

**Note:** Qlik's `[Survey Name] = 'Customer Satisfaction Survey'` filter later in the script relies on the `SurveyName` field — added here too. Field mappings (including CSAT's `Interaction` standardisation and each survey's `MembershipProduct` Hospital Product/Extras fallback) have been cross-checked against `Qualtrics_Data_Extract.md`.

In [ ]:
import numpy as np

SURVEY_NAMES = {
    "CSAT":           "Customer Satisfaction Survey",
    "POC":            "POC Survey",
    "HealthServices": "Health Services",
    "MentalHealth":   "Mental Health Program",
    "Wellbeing":       "UPDATING Health and Wellbeing Program",
    "WellbeingV2":     "HCS NPS Questionnaire (Updated)",
}

# Common output columns, mirroring the Qlik `NPS_Score` load script's field aliases
# NOTE: ProviderType/ProviderAddress/ProviderSuburb/ProviderState/ProviderPostcode require
# joining against Paragon BRONZE tables (Paragon_Provider*, BranchLocations) — out of scope
# here, left as NaN. ProviderNo (via ProgramProviderId_Map) is a static in-script mapping
# table with no BRONZE dependency, so it IS replicated below.
COMMON_COLUMNS = [
    "Survey_id", "SurveyName", "ResponseId", "Interaction", "StartDate", "MonthYear", "EndDate",
    "Provider_Program_Staff", "ProviderType", "ProviderNo", "ProviderAddress", "ProviderSuburb",
    "ProviderState", "ProviderPostcode", "MembershipProduct", "Member Number", "ClaimNo",
    "Postcode", "State", "Region", "CustomerSatisfactionLevel", "CustomerSatisfactionScore",
    "CustomerEffortLevel", "NetPromoterLevel", "NetPromoterScore",
]

# Qlik: ProgramProviderId_Map — a hardcoded `LOAD * Inline` table (Qualtrics_Data_Extract.md
# lines 57-66), NOT sourced from a Paragon BRONZE table, so it's in scope to replicate.
PROGRAM_PROVIDER_ID_MAP = {
    "360 Med Care": "F200017B",
    "Hospital Care at Home": "F300006H",
    "Rehabilitation at Home": "F300006H",
    "MindStep": "F300006H",
    "HWFL": "G200010Y",
}

def program_provider_no(key_series: pd.Series) -> pd.Series:
    # Qlik: ApplyMap('ProgramProviderId_Map', <key>) — unmatched keys map to Null() (ApplyMap's
    # default when no 3rd "default value" argument is given).
    return key_series.map(PROGRAM_PROVIDER_ID_MAP)

def notnull_or_blank(df: pd.DataFrame, col: str) -> pd.Series:
    # Always return a Series aligned to df's index (never a bare scalar), so downstream
    # helpers (blank_to_null, combine_first, etc.) can rely on Series semantics even when
    # the source column is entirely absent from this survey's export.
    if col in df.columns:
        return df[col]
    return pd.Series(np.nan, index=df.index)

def blank_to_null(series: pd.Series) -> pd.Series:
    # Qlik: If(Len(Trim(x))=0, Null(), x) — blank/whitespace-only string treated as Null,
    # not just true NaN
    is_blank = series.isna() | (series.astype(str).str.strip() == "")
    return series.where(~is_blank, np.nan)

def constant_col(value, df: pd.DataFrame) -> pd.Series:
    # A bare scalar assigned to a brand-new column on an empty DataFrame does not
    # broadcast reliably in pandas — build an explicit Series aligned to df's index instead.
    return pd.Series([value] * len(df), index=df.index)

def product_with_extras_fallback(df: pd.DataFrame, product_col: str, extras_col: str = "Extras"):
    # Qlik: If(Len(Trim([Hospital Product]))>0, Trim([Hospital Product]), Trim(Extras)) as Product
    if product_col not in df.columns:
        return pd.Series(np.nan, index=df.index)
    product = df[product_col]
    extras = notnull_or_blank(df, extras_col)
    is_blank = product.isna() | (product.astype(str).str.strip() == "")
    result = product.where(~is_blank, extras)
    return result.astype(str).str.strip().where(result.notna(), result)

def hwfl_substitute(df: pd.DataFrame, col: str = "Survey") -> pd.Series:
    # Qlik: If(Survey='HWFL','Healthy Weight for Life',Survey) as [Provider/Program/Staff]
    survey = notnull_or_blank(df, col)
    return survey.where(survey != "HWFL", "Healthy Weight for Life")

def blank_fallback(primary: pd.Series, fallback: pd.Series) -> pd.Series:
    # Qlik: If(Len(Trim(primary))=0, fallback, primary)
    return blank_to_null(primary).combine_first(fallback)

def qlik_date_only(series: pd.Series) -> pd.Series:
    # Qlik: Date(date#(left(x, 10), 'YYYY-MM-DD')) — truncates to the date portion only,
    # discarding any time-of-day component.
    return pd.to_datetime(series).dt.normalize()

# Qlik: Pick(Match(SubField(Interaction,' ',-2), 'Eye','Dental','Care','Call','Contact'),
#             'Eye Care','Dental','Care Centre','Contact Centre','Contact Centre') as Interaction
_CSAT_INTERACTION_MAP = {"Eye": "Eye Care", "Dental": "Dental", "Care": "Care Centre",
                          "Call": "Contact Centre", "Contact": "Contact Centre"}

def csat_interaction(raw: str):
    if pd.isna(raw) or not str(raw).strip():
        return np.nan
    words = str(raw).strip().split(" ")
    if len(words) < 2:
        return np.nan  # SubField(x,' ',-2) has no second-to-last token -> Match() fails -> Null()
    second_last = words[-2]
    return _CSAT_INTERACTION_MAP.get(second_last, np.nan)

def standardise_csat(df: pd.DataFrame) -> pd.DataFrame:
    out = pd.DataFrame(index=df.index)
    out["Survey_id"] = constant_col(SURVEYS["CSAT"], df)
    out["SurveyName"] = constant_col(SURVEY_NAMES["CSAT"], df)
    out["ResponseId"] = df["ResponseId"]
    out["Interaction"] = df["Interaction"].apply(csat_interaction)
    out["StartDate"] = qlik_date_only(df["StartDate"])
    out["MonthYear"] = pd.to_datetime(df["StartDate"]).dt.strftime("%b %Y")
    out["EndDate"] = qlik_date_only(df["EndDate"])
    # Qlik: If(Len(Trim(operator))>0, operator, CSRName) as [Provider/Program/Staff]
    # An undefined field is Null() in Qlik (Len(Trim(Null()))=0), so a missing `operator`
    # column should still fall back to CSRName, same as a blank value would.
    out["Provider_Program_Staff"] = blank_fallback(notnull_or_blank(df, "operator"), notnull_or_blank(df, "CSRName"))
    out["ProviderType"] = np.nan
    out["ProviderNo"] = np.nan  # Qlik's CSAT LOAD has no ProviderNo field at all
    out["ProviderAddress"] = np.nan
    out["ProviderSuburb"] = np.nan
    out["ProviderState"] = np.nan
    out["ProviderPostcode"] = np.nan
    out["MembershipProduct"] = product_with_extras_fallback(df, "Hospital Product")
    out["Member Number"] = notnull_or_blank(df, "MemberNumber")
    out["ClaimNo"] = np.nan
    out["Postcode"] = notnull_or_blank(df, "Postcode")
    out["State"] = notnull_or_blank(df, "State")
    out["Region"] = notnull_or_blank(df, "Region")
    out["CustomerSatisfactionLevel"] = blank_to_null(notnull_or_blank(df, "Q10_NPS_GROUP"))
    out["CustomerSatisfactionScore"] = blank_to_null(notnull_or_blank(df, "Q10"))
    out["CustomerEffortLevel"] = blank_to_null(notnull_or_blank(df, "Q8_NPS_GROUP"))
    out["NetPromoterLevel"] = blank_to_null(notnull_or_blank(df, "Q2_NPS_GROUP"))
    out["NetPromoterScore"] = blank_to_null(notnull_or_blank(df, "Q2"))
    return out[COMMON_COLUMNS]

def standardise_poc(df: pd.DataFrame) -> pd.DataFrame:
    out = pd.DataFrame(index=df.index)
    out["Survey_id"] = constant_col(SURVEYS["POC"], df)
    out["SurveyName"] = constant_col(SURVEY_NAMES["POC"], df)
    out["ResponseId"] = df["ResponseId"]
    out["Interaction"] = constant_col("Provider of Choice", df)
    out["StartDate"] = qlik_date_only(df["StartDate"])
    out["MonthYear"] = pd.to_datetime(df["StartDate"]).dt.strftime("%b %Y")
    out["EndDate"] = qlik_date_only(df["EndDate"])
    out["Provider_Program_Staff"] = notnull_or_blank(df, "ProviderName")
    out["ProviderType"] = np.nan
    out["ProviderNo"] = notnull_or_blank(df, "ProviderNo")  # Qlik: bare passthrough column
    out["ProviderAddress"] = np.nan
    out["ProviderSuburb"] = np.nan
    out["ProviderState"] = np.nan
    out["ProviderPostcode"] = np.nan
    out["MembershipProduct"] = notnull_or_blank(df, "Product")
    out["Member Number"] = notnull_or_blank(df, "Membership_id")
    out["ClaimNo"] = notnull_or_blank(df, "ClaimNo")
    out["Postcode"] = notnull_or_blank(df, "Postcode")
    out["State"] = notnull_or_blank(df, "State")
    out["Region"] = notnull_or_blank(df, "Region")
    out["CustomerSatisfactionLevel"] = blank_to_null(notnull_or_blank(df, "Q4_NPS_GROUP"))
    out["CustomerSatisfactionScore"] = blank_to_null(notnull_or_blank(df, "Q4"))
    out["CustomerEffortLevel"] = np.nan
    out["NetPromoterLevel"] = blank_to_null(notnull_or_blank(df, "Q6_NPS_GROUP"))
    out["NetPromoterScore"] = blank_to_null(notnull_or_blank(df, "Q6"))
    return out[COMMON_COLUMNS]

def standardise_healthservices(df: pd.DataFrame) -> pd.DataFrame:
    out = pd.DataFrame(index=df.index)
    out["Survey_id"] = constant_col(SURVEYS["HealthServices"], df)
    out["SurveyName"] = constant_col(SURVEY_NAMES["HealthServices"], df)
    out["ResponseId"] = df["ResponseId"]
    out["Interaction"] = constant_col("Health Services", df)
    out["StartDate"] = qlik_date_only(df["StartDate"])
    out["MonthYear"] = pd.to_datetime(df["StartDate"]).dt.strftime("%b %Y")
    out["EndDate"] = qlik_date_only(df["EndDate"])
    out["Provider_Program_Staff"] = notnull_or_blank(df, "Q12")
    out["ProviderType"] = np.nan
    # Qlik: ApplyMap('ProgramProviderId_Map', QID12) as ProviderNo — QID12's export tag is Q12
    out["ProviderNo"] = program_provider_no(notnull_or_blank(df, "Q12"))
    out["ProviderAddress"] = np.nan
    out["ProviderSuburb"] = np.nan
    out["ProviderState"] = np.nan
    out["ProviderPostcode"] = np.nan
    out["MembershipProduct"] = product_with_extras_fallback(df, "HospitalProduct")
    out["Member Number"] = notnull_or_blank(df, "MemberNumber")
    out["ClaimNo"] = np.nan
    out["Postcode"] = notnull_or_blank(df, "Postcode")
    out["State"] = notnull_or_blank(df, "State")
    out["Region"] = notnull_or_blank(df, "Region")
    out["CustomerSatisfactionLevel"] = np.nan
    out["CustomerSatisfactionScore"] = np.nan
    out["CustomerEffortLevel"] = np.nan
    out["NetPromoterLevel"] = blank_to_null(notnull_or_blank(df, "Q2_NPS_GROUP"))
    out["NetPromoterScore"] = blank_to_null(notnull_or_blank(df, "Q2"))
    return out[COMMON_COLUMNS]

def standardise_mentalhealth(df: pd.DataFrame) -> pd.DataFrame:
    out = pd.DataFrame(index=df.index)
    out["Survey_id"] = constant_col(SURVEYS["MentalHealth"], df)
    out["SurveyName"] = constant_col(SURVEY_NAMES["MentalHealth"], df)
    out["ResponseId"] = df["ResponseId"]
    out["Interaction"] = constant_col("Health Services", df)
    out["StartDate"] = qlik_date_only(df["StartDate"])
    out["MonthYear"] = pd.to_datetime(df["StartDate"]).dt.strftime("%b %Y")
    out["EndDate"] = qlik_date_only(df["EndDate"])
    # Qlik: If(Len(Trim(QID22))=0, Survey, QID22) as [Provider/Program/Staff] — QID22's
    # export tag is Q22 (analogous to Wellbeing/WellbeingV2's QID21 -> Q21 pattern).
    provider_program_staff = blank_fallback(notnull_or_blank(df, "Q22"), notnull_or_blank(df, "Survey"))
    out["Provider_Program_Staff"] = provider_program_staff
    out["ProviderType"] = np.nan
    # Qlik: ApplyMap('ProgramProviderId_Map', If(Len(Trim(QID22))=0,Survey,QID22)) as ProviderNo
    out["ProviderNo"] = program_provider_no(provider_program_staff)
    out["ProviderAddress"] = np.nan
    out["ProviderSuburb"] = np.nan
    out["ProviderState"] = np.nan
    out["ProviderPostcode"] = np.nan
    out["MembershipProduct"] = product_with_extras_fallback(df, "HospitalProduct")
    out["Member Number"] = notnull_or_blank(df, "MemberNumber")
    out["ClaimNo"] = np.nan
    out["Postcode"] = notnull_or_blank(df, "Postcode")
    out["State"] = notnull_or_blank(df, "State")
    out["Region"] = notnull_or_blank(df, "Region")
    out["CustomerSatisfactionLevel"] = blank_to_null(notnull_or_blank(df, "Q7_NPS_GROUP"))
    out["CustomerSatisfactionScore"] = blank_to_null(notnull_or_blank(df, "Q7"))
    out["CustomerEffortLevel"] = np.nan
    out["NetPromoterLevel"] = blank_to_null(notnull_or_blank(df, "Q3_NPS_GROUP"))
    out["NetPromoterScore"] = blank_to_null(notnull_or_blank(df, "Q3"))
    return out[COMMON_COLUMNS]

def standardise_wellbeing(df: pd.DataFrame) -> pd.DataFrame:
    out = pd.DataFrame(index=df.index)
    out["Survey_id"] = constant_col(SURVEYS["Wellbeing"], df)
    out["SurveyName"] = constant_col(SURVEY_NAMES["Wellbeing"], df)
    out["ResponseId"] = df["ResponseId"]
    out["Interaction"] = constant_col("Health Services", df)
    out["StartDate"] = qlik_date_only(df["StartDate"])
    out["MonthYear"] = pd.to_datetime(df["StartDate"]).dt.strftime("%b %Y")
    out["EndDate"] = qlik_date_only(df["EndDate"])
    # Qlik: If(Survey='HWFL','Healthy Weight for Life',Survey) as [Provider/Program/Staff]
    out["Provider_Program_Staff"] = hwfl_substitute(df)
    out["ProviderType"] = np.nan
    # Qlik: ApplyMap('ProgramProviderId_Map', Survey) as ProviderNo — keyed on the RAW
    # Survey value (before the HWFL->'Healthy Weight for Life' substitution above).
    out["ProviderNo"] = program_provider_no(notnull_or_blank(df, "Survey"))
    out["ProviderAddress"] = np.nan
    out["ProviderSuburb"] = np.nan
    out["ProviderState"] = np.nan
    out["ProviderPostcode"] = np.nan
    out["MembershipProduct"] = product_with_extras_fallback(df, "HospitalProduct")
    out["Member Number"] = notnull_or_blank(df, "MemberNumber")
    out["ClaimNo"] = np.nan
    out["Postcode"] = notnull_or_blank(df, "Postcode")
    out["State"] = notnull_or_blank(df, "State")
    out["Region"] = notnull_or_blank(df, "Region")
    out["CustomerSatisfactionLevel"] = blank_to_null(notnull_or_blank(df, "Q8_NPS_GROUP"))
    out["CustomerSatisfactionScore"] = blank_to_null(notnull_or_blank(df, "Q8"))
    out["CustomerEffortLevel"] = np.nan
    out["NetPromoterLevel"] = blank_to_null(notnull_or_blank(df, "Q3_NPS_GROUP"))
    out["NetPromoterScore"] = blank_to_null(notnull_or_blank(df, "Q3"))
    return out[COMMON_COLUMNS]

def standardise_wellbeing_v2(df: pd.DataFrame) -> pd.DataFrame:
    # Verified against Qualtrics survey-definitions API + actual data distribution:
    # Q3/Q3_NPS_GROUP = NPS ("On a scale from 0-10, how likely...", values cluster 8-10,
    #   Q3_NPS_GROUP has Promoter/Passive/Detractor); Q4/Q4_NPS_GROUP = Satisfaction
    # ("Overall how satisfied were you..."); Q2 = "Which service did you receive?"
    # (Program Type); Q5 = free-text feedback ("Please provide some information on
    # how we could...").
    out = pd.DataFrame(index=df.index)
    out["Survey_id"] = constant_col(SURVEYS["WellbeingV2"], df)
    out["SurveyName"] = constant_col(SURVEY_NAMES["WellbeingV2"], df)
    out["ResponseId"] = df["ResponseId"]
    out["Interaction"] = constant_col("Health Services", df)
    out["StartDate"] = qlik_date_only(df["StartDate"])
    out["MonthYear"] = pd.to_datetime(df["StartDate"]).dt.strftime("%b %Y")
    out["EndDate"] = qlik_date_only(df["EndDate"])
    # Qlik: If(Survey='HWFL','Healthy Weight for Life',Survey) as [Provider/Program/Staff]
    # (Q2 is "which service did you receive?" — that's [Program Type], a different,
    # out-of-scope field — NOT Provider_Program_Staff.)
    out["Provider_Program_Staff"] = hwfl_substitute(df)
    out["ProviderType"] = np.nan
    # Qlik: ApplyMap('ProgramProviderId_Map', Survey) as ProviderNo — keyed on the RAW
    # Survey value (before the HWFL->'Healthy Weight for Life' substitution above).
    out["ProviderNo"] = program_provider_no(notnull_or_blank(df, "Survey"))
    out["ProviderAddress"] = np.nan
    out["ProviderSuburb"] = np.nan
    out["ProviderState"] = np.nan
    out["ProviderPostcode"] = np.nan
    out["MembershipProduct"] = product_with_extras_fallback(df, "HospitalProduct")
    out["Member Number"] = notnull_or_blank(df, "MemberNumber")
    out["ClaimNo"] = np.nan
    out["Postcode"] = notnull_or_blank(df, "Postcode")
    out["State"] = notnull_or_blank(df, "State")
    out["Region"] = notnull_or_blank(df, "Region")
    out["CustomerSatisfactionLevel"] = blank_to_null(notnull_or_blank(df, "Q4_NPS_GROUP"))
    out["CustomerSatisfactionScore"] = blank_to_null(notnull_or_blank(df, "Q4"))
    out["CustomerEffortLevel"] = np.nan
    out["NetPromoterLevel"] = blank_to_null(notnull_or_blank(df, "Q3_NPS_GROUP"))
    out["NetPromoterScore"] = blank_to_null(notnull_or_blank(df, "Q3"))
    return out[COMMON_COLUMNS]

STANDARDISERS = {
    "CSAT": standardise_csat,
    "POC": standardise_poc,
    "HealthServices": standardise_healthservices,
    "MentalHealth": standardise_mentalhealth,
    "Wellbeing": standardise_wellbeing,
    "WellbeingV2": standardise_wellbeing_v2,
}

In [ ]:
# Build NPS_Score: standardise each survey then concatenate, keeping only rows with a Member Number
standardised = []
for label, df in raw_dfs.items():
    std_df = STANDARDISERS[label](df)
    std_df = std_df[std_df["Member Number"].notna() & (std_df["Member Number"].astype(str).str.strip() != "")]
    print(f"[{label}] {len(std_df):,} rows with a Member Number (of {len(df):,} raw)")
    standardised.append(std_df)

NPS_Score = pd.concat(standardised, ignore_index=True)
print(f"\nNPS_Score total rows: {len(NPS_Score):,}")
NPS_Score.head()

# Export per-survey CSVs (unfiltered, one per survey — mirrors each Qualtrics_*.qvd)
Exports each survey's standardised DataFrame (same `COMMON_COLUMNS` schema as `NPS_Score`) to its own CSV, **before** the Member Number filter is applied — matching Qlik's `Qualtrics_CSAT.qvd` etc., where the `WHERE Len(Trim("Member Number"))>0` filter is only applied later, in `NPS_Score`'s LOAD FROM step, not baked into the QVD itself.

Filenames mirror the Qlik QVD names (`Qualtrics_CSAT.csv`, `Qualtrics_POC.csv`, etc.) and are written to the shared drive `\\prdeqs01\QlikData\Qualtrics`.

In [ ]:
import os

# Qlik QVD naming: Qualtrics_CSAT, Qualtrics_POC, Qualtrics_HealthServices,
# Qualtrics_MentalHealth, Qualtrics_Wellbeing, Qualtrics_Wellbeing_V2
QVD_FILENAMES = {
    "CSAT":           "Qualtrics_CSAT",
    "POC":            "Qualtrics_POC",
    "HealthServices": "Qualtrics_HealthServices",
    "MentalHealth":   "Qualtrics_MentalHealth",
    "Wellbeing":       "Qualtrics_Wellbeing",
    "WellbeingV2":     "Qualtrics_Wellbeing_V2",
}

EXPORT_DIR = r"\\prdeqs01\QlikData\Qualtrics"

for label, df in raw_dfs.items():
    std_df = STANDARDISERS[label](df)  # unfiltered — all rows, including blank Member Number
    out_path = os.path.join(EXPORT_DIR, f"{QVD_FILENAMES[label]}.csv")
    std_df.to_csv(out_path, index=False)
    print(f"[{label}] {len(std_df):,} rows -> {out_path}")

# DataforScreen / NPS_Display (replicates Qlik `DataforScreen` → `NPS_Display.qvd`)
Filters `NPS_Score` to the last 5 financial years (`StartDate >= 1 Jul` of 5 years ago, matching Qlik's `MakeDate(Year(AddYears(today(), -5)), 7, 1)`), then renames fields to match the final Qlik output schema. No `Interaction`/`SurveyName` filter is applied here — that's specific to `DataforBI`, not `DataforScreen`.

In [ ]:
from datetime import date

# Qlik: MakeDate(Year(AddYears(today(), -5)), 7, 1) — 1 Jul, 5 years before today's year
today = date.today()
cutoff = pd.Timestamp(year=today.year - 5, month=7, day=1)
print(f"Cutoff date (StartDate >=): {cutoff.date()}")

NPS_Score["StartDate"] = pd.to_datetime(NPS_Score["StartDate"])

DataforScreen = NPS_Score[NPS_Score["StartDate"] >= cutoff].copy()

DataforScreen = DataforScreen.rename(columns={
    "Member Number": "MembershipNumber",
    "Postcode":      "MemberPostcode",
    "State":         "MemberState",
    "Region":        "MemberBranch",
})

print(f"DataforScreen rows: {len(DataforScreen):,}  (of {len(NPS_Score):,} in NPS_Score)")
DataforScreen.head()